In [162]:
import numpy as np

In [163]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

---

In [164]:
# Forward pass - Python
def forwardPy(inp, W, bias):
    z = 0
    for i in range(len(inp)):
        z += (inp[i] * W[i])
    z += bias
    a = sigmoid(z)
    return z, a

In [165]:
# Forward pass - Numpy
def forwardNp(inp, W, bias):
    z = np.dot(inp, W) + bias
    a = sigmoid(z)
    return z, a

In [166]:
# Test forward pass
num_feats = 2
W = np.random.randn(num_feats)
bias = np.random.randn()
inp = np.random.randn(num_feats)

assert forwardPy(inp, W, bias) == forwardNp(inp, W, bias)

In [167]:
# MSE loss - Python - works only with lists/tuples
def MSELossPy(preds, targs):
    loss = 0
    len_preds = len(preds)
    for i in range(len_preds):
        loss += (preds[i] - targs[i]) ** 2
    loss /= len_preds
    return loss

assert MSELossPy([0, 0], [0, 1]) == 0.5
assert MSELossPy([1, 1], [1, 1]) == 0
assert MSELossPy([0, 0], [1, 1]) == 1

In [168]:
# MSE loss - Numpy - works with numpy arrays, and scalars
def MSELossNp(preds, targs):
    return np.mean((preds - targs) ** 2)

assert MSELossNp(np.array([0, 0]), np.array([0, 1])) == 0.5
assert MSELossNp(np.array([1, 1]), np.array([1, 1])) == 0
assert MSELossNp(np.array([0, 0]), np.array([1, 1])) == 1
assert MSELossNp(0, 0) == 0
assert MSELossNp(0, 1) == 1


---

In [169]:
# XOR dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]) # (4, 2)
y = np.array([0, 1, 1, 0]) # (4,)
num_samples = len(X) # 4

print(X[0])
print(X[:,0])
print(X[0,:])

[0 0]
[0 0 1 1]
[0 0]


In [170]:
# "Model" params (single neuron)
num_features = X.shape[1]
W = np.random.randn(num_features)
bias = np.random.randn()

print(W)
print(bias)

[0.01446788 0.79619926]
0.09239875585598102


In [171]:
z, a = forwardNp(X, W, bias)
loss = MSELossNp(a, y)

print("z:", z)
print("a:", a)
print("y:", y)
print("loss:", loss)

z: [0.09239876 0.88859802 0.10686663 0.90306589]
a: [0.52308327 0.70860077 0.52669126 0.71157914]
y: [0 1 1 0]
loss: 0.272223912301661


In [172]:
# Learning about backprop
dL_da = 2 * (a - y) # (4,)
print("dL_da:", dL_da)

da_dz = a * (1 - a) # (4,)
print("da_dz:", da_dz)

dz_dW = X # (4, 2)
print("dz_dW:", dz_dW)

dL_dz = dL_da * da_dz # (4,)
print("dL_dz:", dL_dz)

dL_dW = np.dot(dz_dW.T, dL_dz) / num_samples # (2, 4) @ (4,) -> (2,)
print("dL_dW:", dL_dW)

# dL_dB is equal to dL_dz

dL_da: [ 1.04616654 -0.58279846 -0.94661748  1.42315827]
da_dz: [0.24946716 0.20648572 0.24928758 0.20523427]
dz_dW: [[0 0]
 [0 1]
 [1 0]
 [1 1]]
dL_dz: [ 0.2609842  -0.12033956 -0.23597998  0.29208085]
dL_dW: [0.01402522 0.04293532]


In [173]:
def backprop(X, y, a):
    dL_da = 2 * (a - y)
    da_dz = a * (1 - a)
    dz_dW = X
    dL_dz = dL_da * da_dz

    num_samples = len(X)
    dL_dW = np.dot(dz_dW.T, dL_dz) / num_samples

    dL_dB = np.mean(dL_dz)
    return dL_dW, dL_dB

In [174]:
# Train the neuron
epochs = 5000
lr = 0.01
for epoch in range(epochs):
    # forward
    z, a = forwardNp(X, W, bias)

    # loss
    loss = MSELossNp(a, y)

    # backward
    gradW, gradB = backprop(X, y, a)

    # update
    W = W - lr * gradW
    bias = bias - lr * gradB

    # print loss every x epochs?
    if epoch % 500 == 0:
        print(f"loss at epoch {epoch}: {loss}")

loss at epoch 0: 0.272223912301661
loss at epoch 500: 0.25824381625863935
loss at epoch 1000: 0.2540734149574988
loss at epoch 1500: 0.25271668315365836
loss at epoch 2000: 0.25203132382857146
loss at epoch 2500: 0.2515618501785606
loss at epoch 3000: 0.25120817216200575
loss at epoch 3500: 0.25093637796828694
loss at epoch 4000: 0.2507268398316646
loss at epoch 4500: 0.25056520067377447


In [175]:
# test
_, a = forwardNp([0,0], W, bias)
print(a)
_, a = forwardNp([0,1], W, bias)
print(a)
_, a = forwardNp([1,0], W, bias)
print(a)
_, a = forwardNp([1,1], W, bias)
print(a)

0.4751307969630826
0.5163118369980616
0.47563423413675576
0.5168159436708465
